In [44]:
import pandas as pd

In [45]:
df = pd.read_csv("data_cleaned_for_early_detection.csv")

In [46]:
TARGET = "Overall Survival Status"

In [47]:
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X.shape, y.shape

((921, 208), (921,))

In [48]:
from sklearn.model_selection import train_test_split

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

X_train.shape, X_val.shape, X_test.shape

((588, 208), (148, 208), (185, 208))

In [50]:
# 1) Identify numeric columns safely:
# convert candidates to numeric and keep only those that become numeric for most rows

X_train_clean = X_train.copy()

numeric_candidates = X_train_clean.columns
num_cols = []
cat_cols = []

for c in numeric_candidates:
    # Try convert to numeric
    converted = pd.to_numeric(X_train_clean[c], errors="coerce")
    # If at least 80% values are numeric (or NaN), treat as numeric
    numeric_ratio = converted.notna().mean()
    if numeric_ratio >= 0.80:
        num_cols.append(c)
        # Replace with numeric version in all splits
        X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
        X_val[c]   = pd.to_numeric(X_val[c], errors="coerce")
        X_test[c]  = pd.to_numeric(X_test[c], errors="coerce")
    else:
        cat_cols.append(c)

print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))

Numeric columns: 3
Categorical columns: 205


In [51]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


In [52]:
num_cols = X_train.select_dtypes(include=["number"]).columns
cat_cols = X_train.select_dtypes(exclude=["number"]).columns

In [53]:
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop"
)

In [54]:
len(num_cols), len(cat_cols)

(96, 112)

In [55]:
import joblib

In [56]:
joblib.dump(preprocess, "data/processed/preprocess_pipeline.pkl")

# Save splits
X_train.to_csv("X_train.csv", index=False)
X_val.to_csv("X_val.csv", index=False)
X_test.to_csv("X_test.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Everything saved successfully.")

Everything saved successfully.


In [57]:
bad_in_num = []
for c in num_cols:
    if X_train[c].dtype == "object":
        bad_in_num.append(c)
bad_in_num[:20], len(bad_in_num)

([], 0)

In [58]:
import joblib
joblib.dump(preprocess, "preprocess_pipeline.pkl")
print("Saved updated preprocess_pipeline.pkl")

Saved updated preprocess_pipeline.pkl
